### 4.2 Elastic Net — regularized, direct multi-horizon

$$\hat\beta_h = \operatorname*{argmin}_{\beta}\ \frac{1}{2n}\lVert y_{t+h}-X_t\beta\rVert_2^2
+ \alpha\rho\lVert\beta\rVert_1 + \alpha\tfrac{1-\rho}{2}\lVert\beta\rVert_2^2 ,\qquad h=1,\dots,8$$

One fitted $\beta_h$ *per horizon* — not one model rolled forward eight times. $\alpha$ and
$\rho$ (`l1_ratio`) are chosen by `GridSearchCV` over chronological `TimeSeriesSplit` folds,
scaler refit inside each fold. The $\ell_1$ term can zero out a weak macro feature entirely;
the $\ell_2$ term keeps correlated survivors from cancelling each other out.

**This argmin does not minimize RMSE, deliberately.** The `(1/2n)‖·‖²` term is proportional
to MSE, but the $\ell_1$/$\ell_2$ penalty terms bias $\hat\beta_h$ *away* from the
MSE-minimizing (OLS) solution on purpose — that trade-off is the entire mechanism of
regularization: accept worse in-sample fit for coefficients that generalize better
out-of-sample. `GridSearchCV`'s own `scoring="neg_mean_squared_error"` isn't RMSE either,
though for *ranking* candidate `(α, l1_ratio)` pairs it's equivalent — RMSE = √MSE, and
√ is monotonic, so MSE and RMSE always pick the same winner. The project's headline RMSE
success criterion (§1) only re-enters once $\hat\beta_h$ is fixed: it's the walk-forward,
out-of-sample metric used in §5 Evaluation to compare Elastic Net's *resulting forecasts*
against SARIMA, Ensemble, seasonal-naive, and the RBA — not a claim that every family's own
training loss literally is RMSE.

*Source: `src/models/elastic_net.py`*

#### Explained, step by step

- Read $\operatorname*{argmin}_\beta [\ldots]$ as "find the coefficient vector $\beta$ that makes the bracketed expression as small as possible."
- $\lVert y_{t+h}-X_t\beta\rVert_2^2$ is the **sum of squared prediction errors** — same idea as RMSE, but not square-rooted or averaged the same way — using today's macro features $X_t$ (unemployment, cash rate, producer prices, etc.) to predict the CPI value $h$ quarters ahead, $y_{t+h}$. Dividing by $2n$ is a scaling convention that simplifies the calculus; it doesn't change which $\beta$ wins.
- $\lVert\beta\rVert_1$ is the sum of the *absolute values* of the coefficients (the **L1 penalty**, or Lasso term). Adding this means every nonzero coefficient "costs" something, so the optimizer will happily set a weak feature's coefficient to *exactly* zero — automatic feature selection.
- $\lVert\beta\rVert_2^2$ is the sum of the *squared* coefficients (the **L2 penalty**, or Ridge term). It discourages any single coefficient from growing too large, and specifically helps when two macro features are correlated (e.g. oil prices and producer prices) — L2 tends to spread the credit between them rather than arbitrarily picking one.
- $\alpha$ controls the overall strength of both penalties; $\rho$ (`l1_ratio` in the code) controls the mix between L1 and Ridge, from $\rho=0$ (pure Ridge) to $\rho=1$ (pure Lasso). Both are chosen automatically via a grid search over chronological `TimeSeriesSplit` folds, so the model is never validated on data from before its own training window.
- A **separate $\hat\beta_h$ is fitted for every horizon** $h=1,\dots,8$ — eight independent models, not one model iterated forward eight times — so the horizon-4 model can lean on entirely different features than the horizon-1 model.
- Why not minimize RMSE directly: the $\frac{1}{2n}\lVert\cdot\rVert_2^2$ term alone would be minimized by plain OLS. Adding the L1/L2 penalties deliberately *pulls the solution away* from that OLS optimum — trading a little in-sample fit for a model that generalizes better out-of-sample. RMSE only re-enters later, in Evaluation, to judge the *resulting forecasts* — not what's being minimized during training.

In [41]:
coefs = pd.read_csv(PROJECT_ROOT / "reports/elastic_net_coefficients.csv")
h1 = coefs[coefs["horizon"] == 1].copy()
h1["abs_coef"] = h1["coef"].abs()
print(f"alpha={h1['selected_alpha'].iloc[0]:.4f}  l1_ratio={h1['selected_l1_ratio'].iloc[0]:.2f}"
      f"  ({(h1['coef'] == 0).sum()} of {len(h1)} horizon-1 features shrunk to exactly zero)")
h1.sort_values("abs_coef", ascending=False).head(6)[["feature", "coef"]]

alpha=0.1668  l1_ratio=0.70  (6 of 11 horizon-1 features shrunk to exactly zero)


,feature,coef
4,inflation_expectations_business_lag1,0.687255
0,cpi_yoy_lag1,0.256226
2,cash_rate_change_lag1,0.122582
7,wti_growth_lag1,0.079576
3,unemployment_rate_change_lag1,-0.003648
1,cpi_yoy_lag4,-0.000000
